In [20]:
import pandas as pd

reviews = pd.read_csv("olist_order_reviews_dataset.csv")
geo = pd.read_csv("olist_geolocation_dataset.csv")

print("Reviews shape:", reviews.shape)
print("Geolocation shape:", geo.shape)


Reviews shape: (99224, 7)
Geolocation shape: (1000163, 5)


In [21]:
reviews['review_comment_message'].isnull().sum()


np.int64(58247)

In [22]:
reviews['review_comment_message'] = reviews['review_comment_message'].fillna("No Comment Provided")


In [23]:
reviews['review_comment_message'].isnull().sum()

np.int64(0)

In [24]:
geo['geolocation_zip_code_prefix'].nunique()


19015

In [25]:
import numpy as np

geo_clean = (
    geo.groupby('geolocation_zip_code_prefix')
       .agg({
           'geolocation_lat': 'mean',
           'geolocation_lng': 'mean',
           'geolocation_city': lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0],
           'geolocation_state': 'first'
       })
       .reset_index()
)


In [26]:
geo_clean.shape


(19015, 5)

In [27]:
reviews['review_comment_message'] = reviews['review_comment_message'].fillna("No Comment Provided")



In [28]:
reviews['review_comment_message'].isnull().sum()

np.int64(0)

In [29]:
reviews.to_csv("reviews_clean.csv", index=False)


In [30]:
geo_clean.to_csv("geolocation_clean.csv", index=False)


In [31]:
import pandas as pd
import numpy as np

# Load cleaned orders CSV
orders = pd.read_csv("orders.csv")

# Quick check
orders.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [32]:
# Columns to convert
date_cols = [
    'order_purchase_timestamp',
    'order_estimated_delivery_date',
    'order_delivered_customer_date'
]

# Convert to datetime
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')


In [33]:
# Difference in days: delivered - estimated
orders['estimated_delivery_variance'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days


In [34]:
orders['delivery_flag'] = np.where(
    orders['estimated_delivery_variance'] <= 0, 'On Time',
    np.where(orders['estimated_delivery_variance'] <= 7, 'Slightly Delayed', 'Severely Delayed')
)


In [35]:
orders.to_csv("orders_enriched.csv", index=False)


In [36]:
# Check counts by category
orders['delivery_flag'].value_counts()

# Spot-check a few rows
orders[['order_id', 'order_estimated_delivery_date', 'order_delivered_customer_date', 
        'estimated_delivery_variance', 'delivery_flag']].head(10)


,order_id,order_estimated_delivery_date,order_delivered_customer_date,estimated_delivery_variance,delivery_flag
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-18,2017-10-10 21:25:13,-8.0,On Time
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-13,2018-08-07 15:27:45,-6.0,On Time
2,47770eb9100c2d0c44946d9cf07ec65d,2018-09-04,2018-08-17 18:06:29,-18.0,On Time
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-15,2017-12-02 00:28:42,-13.0,On Time
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-26,2018-02-16 18:17:02,-10.0,On Time
5,a4591c265e18cb1dcee52889e2d8acc3,2017-08-01,2017-07-26 10:57:55,-6.0,On Time
6,136cce7faa42fdb2cefd53fdc79a6098,2017-05-09,NaT,NaN,Severely Delayed
7,6514b8ad8028c9f2cc2374ded245783f,2017-06-07,2017-05-26 12:55:51,-12.0,On Time
8,76c6e866289321a7c93b82b54852dc33,2017-03-06,2017-02-02 14:08:10,-32.0,On Time
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-23,2017-08-16 17:14:30,-7.0,On Time


In [37]:
orders['delivery_flag'].value_counts()


delivery_flag
On Time             89941
Severely Delayed     5828
Slightly Delayed     3672
Name: count, dtype: int64